# 15 · How SkewerJAX is fast, step by step

Notebook 07 says *what* SkewerJAX computes: the cell crossings of every skewer, each absorbing with its profile
integrated over its line centres, summed into $\tau$. This notebook shows *why it is cheap*, one design element at a
time, each against its naive alternative computed live with the package's own options. It is the visual companion of
[`docs/performance.md`](../docs/performance.md), section by section:

0. **the cost model** $T = T_{\rm fixed} + N\,t_{\rm skewer}$ (§1 of the document);
1. **one pass over the snapshot** (D1): the reach by index arithmetic, what the pass reads, the compact store and the
   tiles, the pipeline;
2. **exact crossings with the profile integrated** (D2): the lower envelope, the integrated profile, accuracy against
   cost;
3. **the windowed engine with far wings by FFT** (D3): the windows, the convolution of the far wings, why not truncate,
   CPU and GPU;
4. **the matched mode on the same route** (D4): chords, the per-cell walk on escalating windows;
5. **fixed shapes and fixed costs** (D5);
6. **why even the matched mode beats fake_spectra**: the two codes' work, stage by stage, measured;
7. **putting it together.**

**Data.** The live demonstrations run on TNG50-4 (snapshot 25, $z = 3$, 19.3 million cells), as notebooks 05 and 07; the
timings at the scale of TNG50-2 (1.2 billion cells) come from the records of notebook 14
(`benchmarks/results/scaling/`) and of notebook 16 (`benchmarks/results/memory/`: per-stage times, CPU utilisation,
bytes read). Runtime: ~3 min.

In [ ]:
import os
import skewerjax.config as config                        # imports neither JAX nor the package: the devices come first
config.configure_devices(n_cpu_devices=2, platforms="cpu")  # the CPU as two JAX devices, as in the benchmarks (docs/performance.md, sec. 5)
import logging, time
logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import use_style, use_backend, add_panel_letters, FIGSIZES
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()        # paper style: Tol's colour-blind-safe "bright" colours and colour maps, minimum font sizes
CPU = config.devices("cpu")

In [ ]:
from skewerjax.core import LineBundle, SkewerBundle, compute_optical_depth, cell_pieces
from skewerjax.core import crossings as cx, tau_engine as te
from skewerjax.core.absorbers import SnapshotFrame, chord_contributions, crossing_contributions
from skewerjax.simulations import ArepoHDF5Reader, snapshot_cells, stream
from skewerjax.comparison import scaling as S, memory as M
from skewerjax.comparison.benchmark import MATCHED_CELLS
from skewerjax.utils.paths import DATA_DIR as DATA, BENCHMARK_RECORDS_DIR   # $SKEWERJAX_DATA, $SKEWERJAX_BENCHMARK_RESULTS
SIM, SNAP, AXIS = "TNG50-4", 25, 2                                  # the skewers along y (axis 2), as in notebooks 12-14
reader = ArepoHDF5Reader(f"{DATA}/{SIM}", SNAP)
H = reader.header
L, AH = H.box_cMpc, H.velfac_kms_per_cMpc                           # [cMpc], (aH) [km/s per cMpc]
nbins, dv = H.pixel_grid(1.0)
v = (np.arange(nbins) + 0.5) * dv                                   # 1 km/s pixels over [0, v_max)
lines = LineBundle(["HI-LyA", "SiIII-1206"], ref_line_id="HI-LyA")
PHYSICS = {"default": S.DEFAULT_CELLS, "matched": MATCHED_CELLS}   # the two physics configurations of the benchmarks
KERNELS = (te._window_values, te._window_kernel, te._cell_kernel, te._deposit_far, te._far_convolve)
n_compiled = lambda: sum(k._cache_size() for k in KERNELS)         # the engine's compiled kernels in this process
R = S.load_records(BENCHMARK_RECORDS_DIR / "scaling")              # notebook 14's records (TNG50-2 timings)
C2 = R.pop("C2", None)
MEM = M.load_memory_records(BENCHMARK_RECORDS_DIR / "memory")      # notebook 16's records (per-stage memory, CPU, bytes read)
print(f"{SIM}: {H.n_gas:,} cells, L = {L:.2f} cMpc, {nbins} pixels of {dv:.4f} km/s; {len(R)} scaling records, {len(MEM)} memory records")

## 0 · The cost model

A code that computes $N$ spectra pays a fixed cost, to read the snapshot and to compute its gas physics, and a cost per
skewer, to find and sum what each skewer crosses ([`docs/performance.md`](../docs/performance.md), §1):

$$
T = T_{\rm fixed} + N\, t_{\rm skewer} .
$$

The figure fits it to the totals of notebook 14's block B (TNG50-2, 16 to 65,536 skewers, cold) for the four entries,
by least squares on the relative residuals.

In [ ]:
from skewerjax.utils.plotting import plot_cost_model
rows_B = S.block_rows(R, "B", S.N_MAX)
N_B = np.array([r["x"] for r in rows_B])
totals = {e: np.array([sum(r["stages"][e].values()) for r in rows_B]) for e in S.ENTRIES}
fits = {e: S.fit_cost_model(N_B, t) for e, t in totals.items()}
pass_s = {e: np.array([r["record"]["pass"]["setup_s"] + r["record"]["pass"]["pass_s"] + r["record"]["pass"]["assemble_s"] for r in rows_B])
          for e in S.SKEWERJAX_ENTRIES}                                     # SkewerJAX's fixed part, measured: the pass
fig, axs = plot_cost_model(N_B, totals, fits, fixed_measured=pass_s, figsize=(22, 8))
fig.suptitle("fake_spectra pays 309 s before its first skewer; SkewerJAX's fixed cost is one read of the snapshot", fontsize=21)
plt.show()
print(f"{'entry':26s} {'T_fixed (fit)':>14s} {'t_skewer (fit)':>15s} {'pass at 16':>11s} {'pass at 65,536':>15s} {'tiles / skewer at 65,536':>25s}")
for e in S.ENTRIES:
    a, b = fits[e]
    p0, p1 = (pass_s[e][0], pass_s[e][-1]) if e in pass_s else (np.nan, np.nan)
    st = rows_B[-1]["stages"][e]
    tile = (st["pieces"] + st["tau"]) / N_B[-1] if e != S.FS else st["tau"] / N_B[-1]
    print(f"{e:26s} {a:12.0f} s {1e3 * b:12.2f} ms {p0:9.0f} s {p1:13.0f} s {1e3 * tile:22.2f} ms")

*Reading.* fake_spectra follows the model exactly: $\sim$309 s at any $N$ (it reads and converts every cell, for each
line) plus 11.6 ms per skewer, so it overtakes its own fixed cost only above $\sim$27,000 skewers. SkewerJAX's fixed part
is the pass (open markers): 26 s at 16 skewers, where it reads the geometry of every cell and the rest of a few rows,
161 s at 65,536, where it reads every page and keeps 58% of the cells. It is not a constant, so its two-term fit is only
indicative (a "fixed" cost of $\sim$34 s, $\sim$4–9 ms per skewer); the per-skewer cost of its tiles, measured at 65,536
skewers, is 3.7 ms (default, CPU), 1.9 ms (GPU) and 6.3 ms (matched), against fake_spectra's 11.6 ms (last column; the
host's crossings run behind the engine). Both terms are smaller: §1 explains the first, §2–§4 the second.

## 1 · One pass over the snapshot (D1)

`simulations.stream` reads the snapshot once, in file order, in blocks of 2 million rows on parallel threads
([`docs/performance.md`](../docs/performance.md), §D1). For each block it reads the geometry of every cell, finds the
cells within reach of some skewer by index arithmetic, reads the other datasets of those rows only, computes their gas
physics and keeps a compact record of each. fake_spectra reads every dataset of every cell, for each line, and converts
them all (§6).

### (a) The reach test by index arithmetic

A cell can be crossed by a skewer only if its generator lies within its reach $R = 2.5\,V^{1/3}$ of the skewer's line.
The skewers run along one axis, so each is a point of the perpendicular plane. The pass buckets them on a grid of
$n_b \times n_b$ squares of that plane ($n_b = \lfloor\sqrt{N}\rfloor$: about one skewer per bucket), and a cell at
$(x, z)$ looks only at the buckets of $[x - R, x + R] \times [z - R, z + R]$: a few distance tests per cell, with no tree
and no sampling points (`core.crossings.line_pairs`). The naive alternative, a k-d tree of the sampling points of every
skewer queried by every cell (the gather route, `select_cells_near_paths`), costs $\sim\log_2$ of the number of points
per cell, and every point must be stored. The figure takes the $64^2$ lattice of TNG50-4 skewers and a few cells of a
thin slab, of increasing size.

In [ ]:
from skewerjax.utils.plotting import plot_reach_test
N_LAT = 64                                                                   # 64 x 64 skewers: one per bucket of 0.81 cMpc
X, Z = S.nested_grid(N_LAT, L)
cofm = np.column_stack([X, np.zeros_like(X), Z])
REGION = ((19.0, 24.0), (19.0, 24.0))                                       # [cMpc] a window of the skewers' plane (x, z)
slab = snapshot_cells(reader, ["HI"], [REGION[0][0], 25.0, REGION[1][0]], [REGION[0][1], 25.3, REGION[1][1]])  # a thin slab in y
size = np.asarray(slab.size_cMpc)
order = np.argsort(size)
xyz = np.asarray(slab.positions)
inside = ((xyz[:, 0] - 2.5 * size > REGION[0][0] + 0.2) & (xyz[:, 0] + 2.5 * size < REGION[0][1] - 0.2)
          & (xyz[:, 2] - 2.5 * size > REGION[1][0] + 0.2) & (xyz[:, 2] + 2.5 * size < REGION[1][1] - 0.6))  # whole circles, room for a label
order = order[inside[order]]
pick = order[(np.array([0.02, 0.35, 0.7, 0.93, 0.995]) * (order.size - 1)).astype(int)]  # from a dense cell to a diffuse one
xy = np.asarray(slab.positions)[pick][:, [0, 2]]
fig, ax, info = plot_reach_test(np.column_stack([X, Z]), L, int(np.sqrt(N_LAT**2)), xy, 2.5 * size[pick], region=REGION, figsize=(22, 9.5))
fig.suptitle("A cell tests only the skewers of the buckets its reach overlaps: index arithmetic, no tree", fontsize=21)
plt.show()
print(f"the {len(pick)} cells (sizes {', '.join(f'{s:.3f}' for s in size[pick])} cMpc): tests {info['tests']}, skewers reached {info['reached']}")

*Reading.* Each star is a cell, its circle its reach. The pass computes, from the cell's coordinates alone, the range of
buckets its reach overlaps (shaded), and tests the distance to the skewers of those buckets only: 2 to 12 for these
cells (the reach of a large, diffuse cell spans several buckets), each reaching one to three skewers. Every cell of the snapshot is
tested this way in a few nanoseconds (the counts of the whole pass are in (b) and §6).

### (b) What the pass reads

The pass reads the geometry (positions, masses, densities: 32 bytes per cell) of every block whole; the other datasets
(velocities, internal energy, electron and H I abundances, metals, star formation: 68 bytes per cell) only for the rows
within reach: as an indexed gather through a memory map when they touch few pages of the block, as one contiguous read
when they touch more than half of them (`stream.GATHER_FRACTION`). The gas physics then runs on those rows only. Here
the pass runs on TNG50-4 for four lattices, and its statistics per block (`stream_cells(...)` info `blocks`) give the
fractions.

In [ ]:
import h5py
from skewerjax.utils.plotting import plot_pass_reads
row_bytes = 0
with h5py.File(reader.files[0], "r") as f:                                  # the bytes per cell of the datasets the codes need
    for k in stream.GEOMETRY + stream.REST:
        ds = f["PartType0"][k]
        row_bytes += ds.dtype.itemsize * int(np.prod(ds.shape[1:]))
lattices, passes = {}, {}
for n in (4, 16, 64, 256):
    Xn, Zn = S.nested_grid(n, L)
    paths_n, frame_n = stream.box_skewers(np.column_stack([Xn, 0 * Xn, Zn]), AXIS, L, H.cosmology(), H.a)
    tile_n = stream.tile_skewers(paths_n, 256 if n == N_LAT else 1024)
    t0 = time.perf_counter()
    store_n, pinfo = stream.stream_cells(reader, paths_n, tile_n, list(lines.species_keys),
                                         PHYSICS if n == N_LAT else {"default": PHYSICS["default"]})
    blocks = pinfo["blocks"]
    rest = lambda b, i: sum(x[i] for k, x in b["read"].items() if k in stream.REST)
    lattices[f"{n}²"] = dict(
        within=pinfo["fraction"], physics=pinfo["fraction"],
        pages=[rest(b, 0) / max(rest(b, 1), 1) for b in blocks], contiguous=[rest(b, 2) > len(stream.REST) / 2 for b in blocks],
        bytes_GB=sum(x[3] for b in blocks for x in b["read"].values()) / 1e9, total_GB=H.n_gas * row_bytes / 1e9)
    passes[n] = (paths_n, frame_n, tile_n, store_n, pinfo)
    print(f"{n:3d}² skewers: pass {time.perf_counter() - t0:.1f} s, {pinfo['n_needed']:,} cells kept ({pinfo['fraction']:.1%}), "
          f"{sum(b['tests'] for b in blocks):,.0f} distance tests ({sum(b['tests'] for b in blocks) / H.n_gas:.2f} per cell), "
          f"{lattices[f'{n}²']['bytes_GB']:.2f} of {lattices[f'{n}²']['total_GB']:.2f} GB read, {sum(lattices[f'{n}²']['contiguous'])} of "
          f"{len(blocks)} blocks read whole")
fig, axs = plot_pass_reads(lattices, figsize=(22, 7.5))
fig.suptitle("The pass reads the geometry of every cell, the rest only where needed, and computes the physics of the needed cells", fontsize=21)
plt.show()
n_pts = N_LAT**2 * nbins                                                    # the naive alternative: sampling points every km/s
print(f"naive search at {N_LAT}²: a k-d tree of {n_pts:,} sampling points, ~{np.log2(n_pts):.0f} node tests per cell "
      f"({H.n_gas * np.log2(n_pts):,.0f} for the snapshot), against {sum(b['tests'] for b in passes[N_LAT][4]['blocks']):,.0f} here")

*Reading.* (a) The cells within reach grow with the number of skewers: a few per cent at $4^2$ and $16^2$, most of the
cells at $64^2$ and nearly all at $256^2$ on this small box (TNG50-4's cells are large: 0.3 cMpc in its voids). Only
those get their gas physics computed; fake_spectra converts every cell, once per line. (b) Each dot is a block of 2
million rows: below the dash-dotted line its other datasets are gathered row by row, above it they are read whole. At
$4^2$ the needed rows touch a few per cent of the pages and are gathered; from $16^2$ on they touch most pages and the
pass reads everything, so its cost stops growing with $N$ (only its store does). (c) The bytes read: the geometry of
every cell (a third of the total) plus the pages of the needed rows. TNG50-2's cells are smaller: the pass reads 47 GB at
16 skewers and every page (119 GB) from $\sim32^2$ skewers on (notebook 16).

### (c) The compact store and the tiles

The pass keeps, for every needed cell, only what the crossings and the contributions need: its position (float64),
size, peculiar velocity and id, and $n$, $T$ of each species for each physics configuration (float32), in one compact
store (`StreamStore`). The skewers are processed in **tiles** of neighbouring skewers (squares of their plane,
`tile_skewers`); the pass records which tiles need each cell, so that each tile gathers its cells by a counting sort,
and a cell near the edge of a tile belongs to both. Here: the $64^2$ lattice in tiles of 256 skewers, with both physics
configurations.

In [ ]:
from skewerjax.utils.plotting import plot_store_tiles
paths, frame, tile_of, store, pinfo = passes[N_LAT]
n_tiles_of = np.bincount(store.tile_cells, minlength=store.n_cells)          # how many tiles need each cell
shared = np.flatnonzero(n_tiles_of >= 2)
sample = np.random.default_rng(1).choice(shared, min(4000, shared.size), replace=False)
fig, axs = plot_store_tiles(np.column_stack([X, Z]), tile_of, L, store.pos[sample][:, [0, 2]], store.nbytes_components(),
                            n_cells=store.n_cells, n_pairs=store.tile_cells.size, figsize=(22, 9))
fig.suptitle("Tiles of neighbouring skewers share few cells; the store keeps 88 bytes per needed cell with two physics configurations", fontsize=21)
plt.show()
print(f"{store.n_cells:,} cells in the store ({store.nbytes / 1e9:.2f} GB); {shared.size / store.n_cells:.1%} of them needed by two tiles "
      f"or more ({store.tile_cells.size / store.n_cells:.2f} tiles per cell); {store.nbytes / store.n_cells:.0f} bytes per cell")

*Reading.* (a) Each colour is a tile of $16 \times 16$ skewers. The ringed cells (a sample) belong to two tiles or more:
those whose reach crosses a tile edge, 3% of the store here. (b) The store holds per cell 48 bytes of geometry (position, size, velocity, id),
16 bytes per physics configuration (two species: $n$ and $T$), and 8 bytes per (cell, tile) pair. A run with one
configuration needs 16 bytes per cell less (notebook 16 measures what this costs at TNG50-2 scale).

### (d) The pipeline

`tiles_optical_depth` hands each tile its cells from the store; three host threads build the crossings and the
contributions of the next tiles while the engine sums the current one. The figure is the timeline of the 16 tiles of
the $64^2$ lattice (default mode, two CPU devices), from the tiles' own clock (`info["rows"]`: `prep_start`,
`prep_end`, `tau_start`, `tau_end`).

In [ ]:
from skewerjax.utils.plotting import plot_tile_timeline
k0 = n_compiled()
taus, tinfo = stream.tiles_optical_depth(store, "default", paths, tile_of, frame, v, lines, pieces="crossings", engines={"tau": CPU})
k1 = n_compiled()
fig, ax, tl = plot_tile_timeline(tinfo["rows"], figsize=(22, 8))
fig.suptitle("The host prepares the next tiles while the engine sums the current one", fontsize=21)
plt.show()
print(f"{tinfo['n_tiles']} tiles in {tl['wall_s']:.2f} s: host preparation {tl['prep_s']:.2f} thread-s, engine {tl['tau_s']:.2f} s "
      f"(sum of the parts / wall = {tl['overlap']:.2f}); {k1 - k0} kernels compiled")

*Reading.* Each tile is two bars: its preparation on a host thread (crossings and contributions; the colour is the
thread) and its sum by the engine (red). The engine starts on tile 0 as soon as it is ready, and from then on the next
tiles are ready before the engine needs them: the host's work (7 thread-s here) runs behind the engine's (13.5 s), so
the tiles cost the engine's time plus the first preparation. The first tiles also compile the kernels (§5), which
stretches the first engine bars. On the GPU the engine is faster and
the host's preparation becomes the bound ([`docs/performance.md`](../docs/performance.md), §7).

## 2 · Exact crossings, with the profile integrated (D2)

### (a) The lower envelope

Along a straight skewer the squared distance to a generator is $(s - u_g)^2 + r_g^2$, and the cell at $s$ is the one of
the lowest line $-2u_g s + u_g^2 + r_g^2$: the lower envelope of the candidates gives every crossing exactly, with no
sampling ([`docs/performance.md`](../docs/performance.md), §D2; notebook 07, §1). The naive alternative,
`deposition="voronoi_segments"`, samples the skewer and cuts the crossings again at the ends of the dual segments of its
points. The figure takes 2 cMpc of skewer 0 of the lattice; the counts are per skewer.

In [ ]:
from skewerjax.utils.plotting import plot_envelope
k_sk = 0
tile0 = np.flatnonzero(tile_of == tile_of[k_sk])
idx = store.tile_cells[store.tile_start[tile_of[k_sk]]:store.tile_start[tile_of[k_sk] + 1]]   # the cells of the skewer's tile
one = paths.take([k_sk])
cr, pairs = cx.straight_crossings(one, store.pos[idx], store.size[idx].astype(np.float64), return_pairs=True)
S0, S1 = 20.0, 22.0                                                             # [cMpc] the segment shown
R_max = 2.5 * store.size[idx].max()
seg = (pairs.u > S0 - R_max) & (pairs.u < S1 + R_max)
in_seg = (cr.s_hi > S0) & (cr.s_lo < S1)
cand = np.flatnonzero(seg)                                                      # the candidates drawn
who = np.array([np.searchsorted(cand, np.flatnonzero((pairs.cell == c) & seg)[0]) for c in cr.cell[in_seg]])  # each crossing's line
counts = {}
cells_tile = store.cells(idx, "default")
for ds_kms in (5.0, 1.0, 0.25):                                                # the option at three spacings
    b1 = SkewerBundle.axis_aligned(cofm[[k_sk]], AXIS, L, ds_kms / AH, H.cosmology(), a_ref=H.a)
    p_opt = cell_pieces(cells_tile, lines.subset([0]), skewer_bundle=b1, deposition="voronoi_segments")
    counts[ds_kms] = int((np.asarray(p_opt.length_m) > 0).sum())
s_pts = np.arange(0.0, L + 1e-9, 5.0 / AH)                                     # the points of the 5 km/s option (drawn)
fig, axs = plot_envelope(pairs.u[seg], pairs.r2[seg], (S0, S1), (cr.s_lo[in_seg], cr.s_hi[in_seg], who),
                         sampled_edges=0.5 * (s_pts[1:] + s_pts[:-1]), counts=dict(crossings=int(cr.counts[0]), sampled=counts[5.0]),
                         figsize=(22, 11))
fig.suptitle("The lower envelope gives every crossing exactly; sampling only adds cuts", fontsize=21)
plt.show()
crs_all = cx.straight_crossings(paths, store.pos, store.size.astype(np.float64))
print(f"skewer {k_sk}: {int(cr.counts[0])} crossings from {pairs.cell.size:,} candidates; the option voronoi_segments: "
      + ", ".join(f"{c:,} pieces at {d:g} km/s" for d, c in counts.items()))
print(f"the {N_LAT}² lattice: {crs_all.counts.mean():.0f} crossings per skewer on average")

*Reading.* (a) Each grey line is a candidate generator within reach of the skewer; at every $s$ the lowest one is the
cell the skewer is in. The envelope (coloured, one colour per crossing) changes cell at its corners $s^*$, the exact
cell boundaries. (b) The crossings are the pieces of the default. The option cuts them again at the ends of the dual
segments of its points (here every 5 km/s for visibility): the same cells, many more pieces, and with the midpoint rule
it needs points every 0.125 km/s for $10^{-6}$ (c): tens of thousands of pieces per skewer instead of a few hundred.

### (b) The integrated profile

A crossing absorbs over the line centres of its span $W$; its profile is the mean of the Voigt function over that span,
$\langle H\rangle = [\Phi(u_{\rm hi}) - \Phi(u_{\rm lo})]/(u_{\rm hi} - u_{\rm lo})$ with $\Phi$ in closed form (erf, Dawson's
function and the higher orders of Harris' expansion; notebook 07, §1). The naive alternative cuts the crossing into $n$
sub-pieces and evaluates the profile at the middle of each (the midpoint rule): it converges as $n^{-2}$. The figure takes
the widest strong $\mathrm{Ly}\alpha$ crossing of skewer 0 ($W/b \approx 2$), with the weight gradient (tilt) it carries.

In [ ]:
from skewerjax.utils.plotting import plot_integrated_profile
contr = crossing_contributions(cr, store.cells(idx, "default"), lines, stream.sub_frame(frame, [k_sk]), one.direction)
c = contr[0]
kappa = 1e4 / float(lines.lambda0_A[0])
b_kms = np.sqrt(2.0) * c.sigma / kappa
a_damp = c.gamma / (np.sqrt(2.0) * c.sigma)
strong = c.amp > np.quantile(c.amp, 0.5)
i_w = np.flatnonzero(strong)[np.argmax((c.width / b_kms)[strong] * (c.width / b_kms < 3)[strong])]  # the widest strong crossing, W < 3 b
fig, axs, info = plot_integrated_profile(float(b_kms[i_w]), float(a_damp[i_w]), float(c.width[i_w]), tilt_kms=float(c.tilt[i_w]),
                                         figsize=(22, 13))
fig.suptitle("The integral over a crossing in closed form: exact at once, where the midpoint rule needs many sub-pieces", fontsize=21)
plt.show()
print("midpoint rule, max error / peak: " + ", ".join(f"{n} sub-pieces {e:.1e}" for n, e in info["midpoint_errors"].items()))
print(f"the tilt term: up to {info['tilt_max']:.1e} of the peak (g = {c.tilt[i_w]:.2e} s/km; on the whole of TNG50-4 it moves Lya by "
      f"rms dF ~2e-6: docs/performance.md, D2)")

*Reading.* (a) The integrated profile (thick) is the profile of a point spread over the span (grey): lower and wider.
The midpoint rule with one sub-piece is the profile of a point at the middle; with more sub-pieces it approaches the
integral. (b) Its error falls as $n^{-2}$: from 46% of the peak with one sub-piece to $1.7\times10^{-4}$ with 32; to reach
the accuracy of Harris' expansion itself ($2\times10^{-7}$) it would need about a thousand sub-pieces for this crossing; the closed form needs one evaluation of $\Phi$ at each end. (c) The terms of
$\Phi$: the erf term carries the Gaussian core, the Dawson term ($\propto a$) the Lorentz wings; the higher orders matter
only for $a$ above $\sim10^{-3}$ (Si III in cold gas). (d) The tilt: $a^{-2}$ and the curvature of the Hubble flow make the
weight of the line centres vary along the span; its first moment adds a term of order $10^{-5}$ of the peak here, which
the default keeps.

### (c) Accuracy against cost

The C2 records of notebook 14: 64 TNG50-2 skewers, the midpoint option on dual-segment pieces sampled every 0.125 to
4 km/s, against the integrated crossings (float64), with the $\tau$ time per skewer of each.

In [ ]:
from skewerjax.utils.plotting import plot_accuracy_cost
fig, axs = plot_accuracy_cost(C2, target=1e-6, figsize=(22, 8))
fig.suptitle("The midpoint rule buys accuracy with pieces; the integrated crossings have it at a fixed cost", fontsize=21)
plt.show()
for k, m in sorted(C2["midpoint"].items(), key=lambda x: float(x[0])):
    print(f"midpoint at {float(k):5.3g} km/s: {m['pieces_per_skewer']:7,.0f} pieces, {1e3 * m['tau_s_per_skewer']:5.1f} ms per skewer, "
          f"Lya rms dF {m['HI-LyA:rms_dF']:.1e}, Si III {m['SiIII-1206:rms_dF']:.1e}")
print(f"integrated crossings: {1e3 * C2['integrated']['tau_s_per_skewer']:.1f} ms per skewer (the reference)")

*Reading.* Each point of the curves is a spacing of the option: finer spacings cost more pieces and more time, and the
error falls as the square of the spacing. Even at 0.125 km/s (33,000 pieces per skewer, 3.2 times the time of the
integrated crossings, the vertical line) Si III's rms is $1.8\times10^{-6}$ and $\mathrm{Ly}\alpha$'s $8.6\times10^{-7}$;
the integrated crossings' own error against the converged
result is $2\times10^{-9}$ in float64 ($4\times10^{-7}$ in float32, its rounding: [`docs/performance.md`](../docs/performance.md),
§D2). The crossings win twice: fewer pieces, and no sampling to converge.

## 3 · The windowed engine, with far wings by FFT (D3)

### (a) The anatomy of $\tau$

The engine evaluates each contribution exactly on a window of pixels $|k - c| \le D$ sized by `wing_tolerance`
($10^{-9}$), and the far Lorentz wings of all the contributions as one convolution of four moments per contribution, by
FFT ([`docs/performance.md`](../docs/performance.md), §D3; notebook 07, step 4). Here on the 256 skewers of tile 0 of the
lattice, where the problem is large enough that the engine takes the windows by itself, in float64
(`jax.experimental.enable_x64`) so that the methods are compared, not float32 rounding; skewer 0 is drawn against the
naive alternative, every crossing on every pixel (the dense layer, `per_piece=True`).

In [ ]:
from skewerjax.utils.plotting import plot_engine_anatomy
sk_t = np.flatnonzero(tile_of == 0)                                          # the 256 skewers of tile 0
b_t = SkewerBundle.axis_aligned(cofm[sk_t], AXIS, L, 0.05, H.cosmology(), a_ref=H.a)   # their paths and frame (a(s) every 0.05 cMpc)
idx0 = store.tile_cells[store.tile_start[0]:store.tile_start[1]]
pieces_t = cell_pieces(store.cells(idx0, "default"), lines, skewer_bundle=b_t)  # the crossings as Pieces (both lines)
lam = np.asarray(lines.lambda0_A, np.float64)
with jax.experimental.enable_x64():
    cj = [te.contributions_from_pieces(pieces_t, j, lines) for j in range(2)]
    plans = [te.window_plan(cj[j], v, len(sk_t), lam[j]) for j in range(2)]
    tau_w = [np.asarray(te.windowed_line_tau(cj[j], v, len(sk_t), lam[j], dtype="float64")) for j in range(2)]
    far = te.far_wing_term(plans[0], len(sk_t), dtype="float64")
    dense0 = np.asarray(compute_optical_depth(pieces_t.skewers(np.array([0])), v, lines, per_piece=True, resolution_tolerance=None,
                                              dtype="float64"))[0, 0]
live0 = (np.asarray(pieces_t.number_density[0, 0]) != 0) & (np.asarray(pieces_t.length_m[0]) > 0)
dense0 = dense0[live0]
r0 = plans[0]["c"].row == 0
plan0 = dict(cpix=plans[0]["cpix"][r0], D=plans[0]["D"][r0], D_x=plans[0]["D_x"])
show = np.argsort(plans[0]["c"].amp[r0])[[-1, r0.sum() // 2, r0.sum() // 8]]
with jax.experimental.enable_x64():                                       # the same kernels on every pixel (no windows)
    full0 = np.asarray(te.windowed_line_tau(cj[0].take(np.flatnonzero(cj[0].row == 0)), v, 1, lam[0], dtype="float64", small_problem=np.inf))[0]
fig, axs, info = plot_engine_anatomy(v, dense0, plan0, far[0], tau_w[0][0], dense0.sum(0), show=show, tau_full=full0,
                                     wing_tolerance=te.WING_TOLERANCE, line="HI-LyA", figsize=(22, 17))
fig.suptitle("Windows sized by the tolerance, the far wings of every contribution in one FFT: the dense sum, exactly", fontsize=21)
plt.show()
print(f"skewer 0: the windows and the FFT against the same kernels on every pixel: max |dtau| {info['full_max_dtau']:.1e}; against the dense sum: max |dtau| {info['max_dtau']:.1e} ({info['rel_dtau']:.0e} of the peak), max |dF| {info['max_dF']:.0e}; "
      f"wings dropped: max |dtau| {info['trunc_max_dtau']:.1e}, max |dF| {info['trunc_max_dF']:.1e}")

*Reading.* As in notebook 07, now with the windows the engine picks by itself: (a) each contribution exact on its window,
(b) the far wings, $\sim10^{-5}$ on every pixel, added by the convolution and equal to the naive sum of the wings beyond
the windows, (c) the engine against the dense sum to $\sim10^{-7}$ of $\tau$ (the two codes form the line centres
separately; against its own kernels on every pixel it agrees to the tolerance, notebook 07), and the windows alone off
by $\sim10^{-5}$.

### (b) Window sizes

The window of a contribution ends where its Gaussian core and the error of its wing series fall below the tolerance: a
strong, broad $\mathrm{Ly}\alpha$ absorber needs a wide window, a weak, narrow Si III one a few pixels. The distributions
are those of every tile of notebook 14's TNG50-2 $128^2$ run (`engine` info of each tile); the evaluations per skewer
compare the dense sum ($N_{\rm contributions} \times N_v$) with the windows ($\sum 2D + 1$; the FFT adds two transforms of
$\sim 2N_v$ points per skewer and line).

In [ ]:
from skewerjax.utils.plotting import plot_window_levels
rec128 = R["TNG50-2_n128_dv1-2-4-8-16"]
tiles128 = rec128["runs"][0]["tiles"][f"{S.CPU_DEF}|1"]
levels, ev = {}, {"dense: every contribution on every pixel": {}, "windows (+ one FFT)": {}}
for j, ln in enumerate(S.LINE_IDS):
    lv = {}
    for t in tiles128:
        for D, n in t["engine"][j]["levels"].items():
            lv[int(D)] = lv.get(int(D), 0) + n
    levels[ln] = lv
    n_c = sum(lv.values()) + sum(t["engine"][j]["n_full"] for t in tiles128)
    ev["dense: every contribution on every pixel"][ln] = n_c * rec128["grids"]["1"]["nbins"] / rec128["config"]["n_skewers"]
    ev["windows (+ one FFT)"][ln] = (sum((2 * D + 1) * n for D, n in lv.items()) + sum(t["engine"][j]["n_full"] for t in tiles128)
                                     * rec128["grids"]["1"]["nbins"]) / rec128["config"]["n_skewers"]
fig, axs = plot_window_levels(levels, ev, figsize=(22, 7.5))
fig.suptitle("Most windows are 64-256 pixels for Lya, 4-16 for Si III: an order of magnitude fewer evaluations", fontsize=21)
plt.show()
for ln in S.LINE_IDS:
    print(f"{ln}: {sum(levels[ln].values()) / rec128['config']['n_skewers']:,.0f} contributions per TNG50-2 skewer; evaluations per skewer "
          f"{ev['dense: every contribution on every pixel'][ln]:,.0f} (dense) against {ev['windows (+ one FFT)'][ln]:,.0f} (windows)")
for j, ln in enumerate(S.LINE_IDS):
    print(f"TNG50-4 tile 0, {ln}: windows {dict(zip(*np.unique(plans[j]['D'], return_counts=True)))}")

*Reading.* (a) $\mathrm{Ly}\alpha$'s windows are mostly 64–256 pixels: its crossings are strong ($\tau_0 \sim 1$) and
their Gaussian cores must fall to $10^{-9}$ inside the window. Si III's are 4–16 pixels: its optical depths are
$10^{-6}$ of $\mathrm{Ly}\alpha$'s. (b) The dense sum evaluates every contribution on all 3,961 pixels; the windows need
10–20 times fewer evaluations for $\mathrm{Ly}\alpha$ and hundreds of times fewer for Si III, and the far wings cost one FFT
per skewer and line, whatever the number of contributions.

### (c) The far wings as one convolution

Beyond its core every profile is its Lorentz wing, $w/\Delta'^2\,[1 + \tfrac32 b^2/\Delta'^2]$, the same function of the
offset for every contribution up to four weights. So each contribution deposits four moments at its centre pixel, and
the far term of all of them is the convolution of the deposits with four kernels, zero within $D_x$ pixels. The figure
compares the FFT convolution (`tau_engine.far_wing_term`) with the direct sum of the same series over the contributions
of skewer 0.

In [ ]:
from skewerjax.utils.plotting import plot_far_convolution
p0 = plans[0]
w0 = np.flatnonzero(~p0["full"] & (p0["c"].row == 0))                       # the windowed contributions of skewer 0
M2 = 2 * nbins                                                                  # offsets to +-2 N_v: the centres may lie off the grid
offs = np.arange(-M2, M2 + 1) * dv
with jax.experimental.enable_x64():
    kern = np.stack([np.asarray(k, np.float64) for k in te.far_kernels(jnp.asarray(offs), "exact")])
kern[:, np.abs(np.arange(-M2, M2 + 1)) <= p0["D_x"]] = 0.0                    # zero within D_x pixels
j = np.arange(nbins)
direct = sum(p0["m"][q][w0] @ kern[q][(j[None, :] - p0["cpix"][w0][:, None]) + M2] for q in range(4))   # the series, contribution by contribution
fig, axs, info = plot_far_convolution(v, p0["cpix"][w0].clip(0, nbins - 1), p0["m"][:, w0], offs[M2:], kern[:, M2:], far[0], direct,
                                      D_x=p0["D_x"], figsize=(22, 15))
fig.suptitle("Four moments per contribution, one convolution per skewer: the far wings of every contribution at once", fontsize=21)
plt.show()
print(f"{w0.size} windowed contributions on skewer 0; the FFT term against the direct sum: max {info['max_diff']:.1e} "
      f"({info['rel_diff']:.0e} of the term); directly: {4 * w0.size * nbins:,} kernel evaluations")

*Reading.* (a) Four numbers per contribution, at its centre pixel: $w$ (the Lorentz weight, $\propto$ its column),
its offset from the pixel, its spread over the span, and its Doppler width. (b) The kernels fall as $\Delta^{-2}$ and
faster, the same for every contribution; they are zero within $D_x$, where every contribution is inside its own window.
(c) The convolution reproduces the direct sum to rounding, at the cost of an FFT instead of $N_{\rm contributions}
\times N_v$ evaluations.

### (d) Why not truncate

The cheapest alternative would drop the wings beyond the windows. On the first 16 skewers of tile 0: the error of the
windows alone and of the engine against the dense sum, on every pixel.

In [ ]:
from skewerjax.utils.plotting import plot_truncation, engine_parts_summary
dt_tr, dF_tr, dt_en, dF_en = [], [], [], []
with jax.experimental.enable_x64():
    for k in range(16):
        dk = np.asarray(compute_optical_depth(pieces_t.skewers(np.array([k])), v, lines, per_piece=True, resolution_tolerance=None,
                                              dtype="float64"))[0, 0]
        dk = dk[(np.asarray(pieces_t.number_density[0, k]) != 0) & (np.asarray(pieces_t.length_m[k]) > 0)]
        rk = plans[0]["c"].row == k
        s_k = engine_parts_summary(v, dk, dict(cpix=plans[0]["cpix"][rk], D=plans[0]["D"][rk]), far[k], tau_w[0][k], dk.sum(0))
        F = lambda t: np.exp(-t)
        dt_tr.append(s_k["truncated"] - dk.sum(0)); dF_tr.append(F(s_k["truncated"]) - F(dk.sum(0)))
        dt_en.append(tau_w[0][k] - dk.sum(0)); dF_en.append(F(tau_w[0][k]) - F(dk.sum(0)))
fig, axs = plot_truncation(dt_tr, dF_tr, dt_en, dF_en, wing_tolerance=te.WING_TOLERANCE, target=1e-6, figsize=(22, 7))
fig.suptitle("Dropping the far wings errs by 1e-5 in tau and in F on every pixel, ten times the flux target", fontsize=21)
plt.show()
rms = lambda x: float(np.sqrt(np.mean(np.square(x))))
print(f"16 skewers, Lya: wings dropped rms dF {rms(dF_tr):.1e} (max {np.abs(dF_tr).max():.1e}); the engine rms dF {rms(dF_en):.1e} "
      f"(max {np.abs(dF_en).max():.1e})")

*Reading.* Without the far wings every pixel misses the Lorentz wings of the whole forest: $\sim10^{-5}$ in $\tau$
(a), an rms $\Delta F$ of $2\times10^{-5}$ (b), twenty times the $10^{-6}$ the package aims at. With them the engine sits at
the agreement of the two codes, orders of magnitude below. The far term costs $\sim1\%$ of the engine's time.

### (e) CPU against GPU

The kernels are the same on both. On the CPU, XLA's scatter-add is serial, so the kernels return each contribution's
window and the host adds them, in parallel over skewers; the CPU is exposed as two JAX devices so that two kernels
overlap and the cores stay busy. On a GPU the device scatter-adds the windows with atomics
([`docs/performance.md`](../docs/performance.md), §5). The $\tau$ stage of notebook 14:

In [ ]:
for r in rows_B[-2:]:
    st, n = r["stages"], r["x"]
    print(f"{n:6,d} TNG50-2 skewers: tau {1e3 * st[S.CPU_DEF]['tau'] / n:.2f} ms per skewer on the CPU (two devices), "
          f"{1e3 * st[S.GPU_DEF]['tau'] / n:.2f} ms on the GPU (incl. the engine's host side); "
          f"matched {1e3 * st[S.CPU_MAT]['tau'] / n:.2f} ms")

## 4 · The matched mode on the same route (D4)

The matched mode reproduces fake_spectra's own calculation (notebook 12): every cell a sphere of radius
$h = V^{1/3}$ crossing the line along a chord sampled at 7 points, fake_spectra's frame and constants, and each cell
walking out from its centre pixel until it adds less than `tautail` $= 10^{-7}$
([`docs/performance.md`](../docs/performance.md), §D4). It runs on the same pass and tiles as the default, from the
second physics configuration of the store.

### (a) Chord geometry

The chord is straight and the Hubble flow linear in fake_spectra's frame, so the 7 points are equally spaced in line
centre: a chord is its middle line centre, its spacing and its amplitude, one contribution, and the per-cell kernel
generates the 7 points on the device (`LineContributions.spacing`).

In [ ]:
from skewerjax.utils.plotting import plot_chord
mt = S.mode_options("matched", nbins, dv, H)["tau"]
snap = SnapshotFrame(H.a, AH, path_length=np.full(1, L))
cells_m = store.cells(idx, "matched")
ch = cx.straight_chords(one, store.pos[idx], store.size[idx].astype(np.float64))
cm = chord_contributions(ch, cells_m, lines, snap, one.direction, one.length, L, chord_weight=mt["chord_weight"], a_eff=mt["a_eff"],
                         b_constants=mt["b_constants"], hubble_flow=mt["hubble_flow"], aH_ref=mt["aH_ref"])
h_c = store.size[idx][ch.cell].astype(np.float64)
r_c = np.sqrt(np.maximum(h_c**2 - ch.z**2, 0.0))
tau0 = cm[0].amp / (np.sqrt(2 * np.pi) * cm[0].sigma)
i_c = np.argmax(tau0 * (r_c > 0.4 * h_c) * (r_c < 0.7 * h_c))                    # a strong chord at a middling impact parameter
bm = np.sqrt(2.0) * cm[0].sigma / kappa
fig, axs = plot_chord(float(h_c[i_c]), float(r_c[i_c]), float(bm[i_c]), float(cm[0].gamma[i_c] / (np.sqrt(2) * cm[0].sigma[i_c])),
                      float(cm[0].spacing[i_c]), tau0=float(tau0[i_c]), figsize=(22, 8))
fig.suptitle("fake_spectra's chord: 7 points equally spaced in line centre, generated on the device from one contribution", fontsize=21)
plt.show()
print(f"skewer {k_sk}: {ch.cell.size} chords (spheres touching the line), {int(cr.counts[0])} crossings; the host prepares {ch.cell.size} rows, "
      f"not {7 * ch.cell.size}")

*Reading.* (a) The sphere of the cell (grey), the path at impact parameter $r$, and the chord's 7 points (fake_spectra's
quadrature, `singleabs.h`). (b) Their profiles, each with a seventh of the column, and their sum: the kernel builds them
from the chord's middle line centre and spacing, so the host prepares one row per chord instead of seven.

### (b) The per-cell walk

fake_spectra adds each cell pixel by pixel from its centre outwards, and stops on each side at the first pixel where
the cell adds less than `tautail`. SkewerJAX evaluates the cell on a window around its centre pixel, sized from its peak
and its damping wing, finds the same stopping pixels by a reduction, and escalates to the next window level when the walk
reaches the window's edge; beyond 10 Doppler widths from its extreme points the cell is evaluated once per pixel from
four moments (its Lorentz wing) instead of 7 times. Here for the chord of skewer 0 with the narrowest window split into
core and wings (one that escalated, if any did; the counts are printed), against the same walk on the whole grid
(`levels` of one level of $N_v$ pixels).

In [ ]:
from skewerjax.utils.plotting import plot_cell_walk
kw = dict(period=np.full(1, nbins * dv), profile="midpoint", detuning="linear", pixel_average=True)
_, all_info = te.per_cell_line_tau(cm[0], v, 1, lam[0], tautail=1e-7, return_info=True, **kw)   # every chord of the skewer
esc = all_info["level_end"] > all_info["level_start"]                                      # the walks that escalated
split = (all_info["core"] > 0) & (all_info["level_end"] >= 0)                               # the windows split into core and wings
pick = esc & split if (esc & split).any() else (esc if esc.any() else split)
i_s = int(np.flatnonzero(pick)[np.argmin(all_info["level_end"][pick])])                    # the narrowest such window
print(f"{len(cm[0])} chords on skewer 0: {int(esc.sum())} walks escalated, {int(split.sum())} windows split into core and wings")
one_cell = cm[0].take([i_s])
t_walk, w_info = te.per_cell_line_tau(one_cell, v, 1, lam[0], tautail=1e-7, return_info=True, **kw)
t_full_walk = te.per_cell_line_tau(one_cell, v, 1, lam[0], tautail=1e-7, levels=(nbins,), **kw)
t_profile = te.per_cell_line_tau(one_cell, v, 1, lam[0], tautail=None, **kw)
c0 = int(w_info["centre"][0])
roll = lambda t: np.roll(np.asarray(t, np.float64)[0], nbins // 2 - c0)        # centred on the cell's pixel
offs_px = np.arange(nbins) - nbins // 2
fig, axs, info = plot_cell_walk(offs_px, roll(t_profile), roll(t_walk), roll(t_full_walk), tautail=1e-7, levels=w_info["levels"],
                                level_start=int(w_info["level_start"][0]), level_end=int(w_info["level_end"][0]), core=int(w_info["core"][0]),
                                b_px=float(bm[i_s] / dv), figsize=(22, 10))
fig.suptitle("fake_spectra's walk, computed on a window and escalated only when it reaches the edge", fontsize=21)
plt.show()
print(f"tau0 = {tau0[i_s]:.2f}: windows {[int(w_info['levels'][L]) for L in range(int(w_info['level_start'][0]), int(w_info['level_end'][0]) + 1)]} px, "
      f"the walk keeps {info['walk_px']} pixels; core {int(w_info['core'][0])} px each way; against the full-grid walk: max |dtau| "
      f"{info['max_diff']:.0e}")

*Reading.* (a) The cell's profile on every pixel (dotted), the walk (blue: the pixels it keeps, up to the first one below
`tautail` on each side), and the full-grid walk underneath: identical. The shaded core is evaluated point by point (7
points per pixel); beyond it, where every point is in its Lorentz wing, one evaluation per pixel. (b) The windows
tried: the first one, estimated from the peak and the damping wing, and the next ones only if the walk reached the edge.
The matched mode does fake_spectra's work in the same order, without visiting the pixels one by one.

## 5 · Fixed shapes and fixed costs (D5)

Every kernel has fixed shapes: the rows of a tile are padded to a power of two, the chunks of contributions have a fixed
size per window level (the last one padded with empty contributions), and the far-wing domain is fixed per grid. So a
kernel compiles once per window level, kind and Voigt order, whatever the tile ([`docs/performance.md`](../docs/performance.md),
§D5). Here two more lattices, $90^2$ and $100^2$ skewers in tiles of about 1,024 (900 and 625 skewers per tile), after the
runs above; and the JIT times of notebook 14's records.

In [ ]:
from skewerjax.utils.plotting import plot_fixed_shapes
sizes, new = [], []
for n in (90, 100):
    Xn, Zn = S.nested_grid(n, L)
    pn, fn = stream.box_skewers(np.column_stack([Xn, 0 * Xn, Zn]), AXIS, L, H.cosmology(), H.a)
    tn = stream.tile_skewers(pn, 1024)
    sn, _ = stream.stream_cells(reader, pn, tn, list(lines.species_keys), {"default": PHYSICS["default"]})
    before = n_compiled()
    _, ti = stream.tiles_optical_depth(sn, "default", pn, tn, fn, v, lines, engines={"tau": CPU})
    new.append(n_compiled() - before)
    sizes += [r["n_skewers"] for r in ti["rows"]]
    del sn
seen = {256, 900, 625}                                                          # the tile sizes of this notebook (sections 1d and 5)
compiled = {"compiled in this notebook": n_compiled(), "unpadded: one set per tile size, at least": n_compiled() * len(seen)}
jit = {f"{r['x']:,} TNG50-2 skewers": r["record"]["jit_s"] for r in rows_B}
fig, axs = plot_fixed_shapes(sizes, compiled, jit, figsize=(22, 7.5))
fig.suptitle("Padded shapes: tiles of 576 to 900 skewers share the kernels of 1,024 rows", fontsize=21)
plt.show()
print(f"tile sizes {sorted(set(sizes))} -> rows {sorted({1 << (s - 1).bit_length() for s in sizes})}; new kernels compiled by the two lattices: {new}; "
      f"{n_compiled()} kernels in this process")

*Reading.* (a) Tiles of 576 to 900 skewers run on kernels of 1,024 rows: one shape. The first of the two lattices
compiled the kernels of that row count (§1d's tiles had 256 rows), 11 of them; the second only 1 (a window level not met
before). (b) The engine's kernels compiled in this notebook, every level and kind it met, against the sets that each
distinct tile size would compile anew without padding (at least). (c) The JIT compilation of a benchmark point (17–31 s, notebook 14; 99 s at 16,384 skewers, which compiles for five pixel
grids) is a fixed cost paid once per process: nothing
is stored on disk between runs (no JAX compilation cache, numba without `cache=True`, no snapshot data), so a cold run is
a first run, and its time includes everything but this compilation.

## 6 · Why even the matched mode beats fake_spectra

The matched mode computes fake_spectra's physics with fake_spectra's algorithm (spheres, chords of 7 points, the linear
flow, `tautail`, `pixel_average`; agreement $\sim10^{-7}$ in $F$, notebook 12), yet it is 2.8 times faster at $128^2$ TNG50-2
skewers, 1.9 times at $256^2$ and 12 times at 16 skewers (notebook 14). Where does the difference come from?

### What fake_spectra does

From its code (`MAT/fake_spectra/fake_spectra/`, the copy installed in `skewerjax-fs`, upstream `ff5844d`):

- **Reading** (`spectra.py:Spectra.compute_spectra`, L838–868, calling `_interpolate_single_file`, L530): one file at a time,
  every cell of it, every dataset it needs read whole through h5py in the main thread
  (`abstractsnapshot.py:HDF5Snapshot.get_data`, L209–224), no spatial selection. **Each line repeats it all**: H I and
  Si III each read every dataset again (`Density` three times per file for H I), $\sim$100 bytes per cell per line.
- **Gas physics** (`gas_properties.py`, `abstractsnapshot.py:get_temp`): vectorised NumPy, single-threaded, on **every
  cell** of each file, once per line (H I: the stored fraction with the Rahmati correction of star-forming cells;
  Si III: the CLOUDY table by cubic `map_coordinates`); the arrays are then cast to float64 for C (`spectra.py`,
  L711–716).
- **Search** (`index_table.cpp:IndexTable::get_near_particles`, L274–338): a mesh of $\lfloor\sqrt{N}\rfloor^2$ cells
  over the sightlines (the same index arithmetic as SkewerJAX's), rebuilt per file and line; **every cell of the file** is
  tested against the sightlines of the mesh cells its sphere (radius $h$) overlaps, in parallel (OpenMP, static
  chunks); per-thread lists, a serial prefix sum, a parallel scatter.
- **C engine** (`part_int.cpp:ParticleInterp::compute_tau`, L75–105; `absorption.cpp`, L215–282; `singleabs.h`):
  OpenMP over sightlines (default static schedule); for each particle near a sightline, the walk from its pixel outwards
  until `tautail`, each pixel 7 chord points times 1 (H I) or $\ge 3$ (cold Si III, the pixel average) sub-pixel points;
  the profile by a tabulated Harris expansion for $a \le 10^{-3}$ and the complex Faddeeva function otherwise (every
  Si III evaluation); double precision, compiled `-O2 -ffast-math`. The GIL is held, and the files are processed one
  after the other: the read and the physics of a file never overlap the C call of the previous one.

### Measured

The records of notebook 16 (one fresh process per code, TNG50-2, $128^2$ skewers, cold) sample every 0.05 s the CPU time
of the process (busy cores) and the bytes it reads from the disk, with the stages marked. A third run,
`fake_spectra, search only`, sets `tautail` = $10^{30}$: every particle stops after one pixel, so its C call is the search
(and a negligible engine).

In [ ]:
from skewerjax.utils.plotting import plot_cpu_profile
key = lambda n, e: M.point_id("TNG50-2", n, 1.0, e)
ids = {lab: key(128, e) for lab, e in (("fake_spectra", S.FS), ("SkewerJAX matched", S.CPU_MAT), ("search", M.FS_SEARCH))}
have6 = all(i in MEM for i in ids.values())
if have6:
    fs, sm, fss = (MEM[ids[k]] for k in ("fake_spectra", "SkewerJAX matched", "search"))
    tl = {}
    for lab, r in (("fake_spectra", fs), ("SkewerJAX matched, CPU", sm)):
        spans = M.stage_spans(r)
        t, c = M.busy_cores(r["timeline"])
        t0 = spans[0][0]
        keep = t >= t0
        tl[lab] = (t[keep] - t0, c[keep], [(a - t0, b - t0, n) for a, b, n in spans])
    gap = M.gap_table(fs, sm, fss)
    stg = {g["stage"]: {"fake_spectra": g["fs_s"] if np.isfinite(g["fs_s"]) else 0.0, "SkewerJAX matched, CPU": g["sjx_s"]} for g in gap}
    fig, axs = plot_cpu_profile(tl, stg, cores=32, figsize=(22, 12))
    fig.suptitle("fake_spectra keeps 1.5 cores busy on average; SkewerJAX keeps 11-15 busy, and does less work before its engine", fontsize=21)
    plt.show()
else:
    print("the memory records of notebook 16 are missing: run scripts/benchmarks/benchmark_memory.py (or notebook 16 with RUN = True)")

*Reading.* (a) fake_spectra alternates, file after file, a serial read (yellow), a serial gas physics (purple) and a
parallel C call (red: search and engine), once for H I and once for Si III; for most of the run one core is busy. (b)
SkewerJAX's pass keeps the cores busy while it reads (the I/O, the reach and the physics of different blocks overlap),
then the tiles keep them busy with the engine. (c) The stages side by side: every one is shorter in SkewerJAX.

### The gap, stage by stage

The work of each code is counted from the records: the bytes read (measured), the cells whose gas physics is computed,
the distance tests of the search (fake_spectra's, with its reach $h$, counted by SkewerJAX's pass on the same cells),
the chord points prepared on the host, the profile evaluations (fake_spectra's walk estimated from each cell's profile:
`per_cell_line_tau`'s `walk_evaluations`). The ratio of the times splits, in logarithm, into the ratio of the work
(**algorithmic**: less work), the ratio of the busy cores (**hardware use**) and the rest (**implementation**: the same
work per core done faster), and each stage's gap is shared in proportion.

In [ ]:
if have6:
    print(f"{'stage':17s} {'fake_spectra':>12s} {'matched':>8s} {'ratio':>6s}  {'work: fs / SkewerJAX':>24s}  {'busy cores':>11s}  "
          f"{'algorithmic':>11s} {'hardware use':>12s} {'implementation':>14s}")
    for g in gap:
        sh = g["shares"]
        f = lambda x: "-" if not np.isfinite(x) else f"{x:.0f} s"
        print(f"{g['stage']:17s} {g['fs_s']:10.1f} s {g['sjx_s']:6.1f} s {g['ratio']:6.2f}  {g['work_fs']:11.3g} / {g['work_sjx']:<10.3g}  "
              f"{g['cores_fs']:4.1f} / {g['cores_sjx']:<4.1f}  {f(sh['algorithmic']):>11s} {f(sh['hardware use']):>12s} {f(sh['implementation']):>14s}")
    tot = {k: sum(g["shares"][k] for g in gap if np.isfinite(g["shares"][k])) for k in ("algorithmic", "hardware use", "implementation")}
    T_fs, T_sm = sum(fs["stages"].values()), sum(g["sjx_s"] for g in gap)
    print(f"total: fake_spectra {T_fs:.0f} s, matched {T_sm:.0f} s (the stages above; + {sm['pass_info']['assemble_s']:.0f} s of store assembly); "
          f"the gap {T_fs - T_sm:.0f} s: " + ", ".join(f"{k} {v:.0f} s ({v / (T_fs - T_sm):.0%})" for k, v in tot.items()))
    print(f"busy cores over the whole run: fake_spectra {M.mean_busy_cores(fs, ('io', 'physics', 'tau')):.1f}, "
          f"matched SkewerJAX {M.mean_busy_cores(sm, ('pass', 'assemble', 'tiles')):.1f}")
    t = gap[-1]
    print(f"tau accumulation, core-ns per profile evaluation: fake_spectra {1e9 * t['fs_s'] * t['cores_fs'] / t['work_fs']:.1f}, "
          f"SkewerJAX {1e9 * t['sjx_s'] * t['cores_sjx'] / t['work_sjx']:.1f}")
    for n in (4, 128):
        f_, s_ = MEM.get(key(n, S.FS)), MEM.get(key(n, S.CPU_MAT))
        if f_ and s_:
            print(f"\n{n * n:,} skewers, work fake_spectra / matched SkewerJAX:")
            for k, (a, b) in M.work_counts(f_, s_).items():
                print(f"  {k:36s} {a:12.4g} / {b:<12.4g} ({a / b:.3g}x)")

*Reading.* The gap of $\sim$380 s at $128^2$ TNG50-2 skewers splits into (the numbers above; the method:
`comparison.memory.gap_table`):

- **algorithmic, $\sim$30%**: less work. fake_spectra reads and converts every cell for each line; SkewerJAX reads each
  byte once (1.4 times fewer bytes here: the page cache spares fake_spectra part of its second read), computes the
  physics of the 23% of the cells within reach (4.4 times fewer cell-species), and tests each cell once against the
  skewers' buckets, during the pass.
- **hardware use, $\sim$50%**: fake_spectra keeps 1.5 cores busy on average over its run: its reads and its NumPy physics are
  serial, and its C calls, file after file, average 2.9 busy cores (each call is short and has serial parts: the casts to
  float64, the index table's prefix sums, the static schedule over sightlines). SkewerJAX's pass keeps $\sim$11 cores
  busy and its tiles $\sim$15.
- **implementation, $\sim$20%**: the same work done faster: in the pass, parallel block reads (the disk delivers
  $\sim$3 GB/s to 32 threads against $\sim$0.75 GB/s to fake_spectra's serial h5py reads) and the fused physics loops
  (in the pass the use of the cores cannot be told apart from the implementation: its stages overlap on shared
  threads). In the $\tau$ stage the implementation share is *negative*: per core and per profile evaluation fake_spectra's
  C engine (scalar double precision, a tabulated Harris expansion, `-O2 -ffast-math`) is $\sim$4 times faster than
  SkewerJAX's windows through XLA; SkewerJAX wins that stage only by keeping 5 times more cores busy.

So even the matched mode wins mostly by doing the work before the engine once, in parallel, on the cells that matter, and
by keeping the machine busy; its engine is no faster per core.

### What limits fake_spectra, and what it would need

Ranked by the time each costs it at $128^2$ TNG50-2 skewers (the table above):

1. **Reading every cell, for each line, serially** (216 s: 163 GB at $\sim$0.75 GB/s, half a core busy): read each
   dataset once for all lines, in parallel blocks, and only the rows near the sightlines (a reach test on the positions
   first).
2. **The gas physics of every cell, for each line, on one core** (147 s): compute it for the cells near a sightline only,
   once, on many threads.
3. **The C engine at 2.9 busy cores** (130 s): its per-core speed is good; it needs the cores. Batching several files
   (or all the particles near the sightlines) per call, a dynamic schedule and parallel casts would keep the 32 threads
   busy; a core / wing split would also cut its evaluations in the far wings.
4. **The search, per file and per line** (70 s with the casts to float64): one index of the sightlines, the particles
   tested once, in the reading pass.
5. **No overlap**: the read and the physics of a file never overlap the C call of the previous one; a pipeline would
   hide the shorter of them.

### What this implies for SkewerJAX

- **The disk is the floor** of any code that reads the snapshot: TNG50-2's needed datasets, 119 GB read by the pass
  (measured), at $\sim$2.5 GB/s cold, $\sim$48 s. The pass at $128^2$ takes 70–81 s: the reach test and the physics
  overlap the read, and $\sim$25–30 s remain above the floor (the reach and pair bookkeeping, the gathers' page faults).
- **The matched mode's engine** costs 6.3 ms per skewer against the default engine's 3.7 (notebook 14): it exists to
  reproduce fake_spectra. Its per-core speed is $\sim$4 times below fake_spectra's C engine (the table): vectorising the
  walk better (fewer padded window pixels, the 7 points of a chord evaluated together in the core) could gain up to
  $\sim$2 ms per skewer.
- **The host's preparation** of the tiles (crossings, contributions: $\sim$4 thread-ms per skewer) bounds the GPU entry
  (1.9 ms per skewer): computing them on the device would bring it towards the pass.
- **Memory** grows with the cells within reach (notebook 16): one physics configuration per pass and float32 positions
  relative to the tile would cut the store by a third.

## 7 · Putting it together

Where the time goes at a few numbers of skewers (notebook 14's block B, TNG50-2, cold), and what each design element
saves per skewer, against its naive alternative.

In [ ]:
from skewerjax.utils.plotting import plot_stage_grid
pick_B = [r for r in rows_B if r["x"] in (16, 1024, 16384, 65536)]
fig, axs = plot_stage_grid([r["stages"] for r in pick_B], labels=[r["title"] for r in pick_B], jit=[r["jit"] for r in pick_B])
add_panel_letters(axs, labels=[f"{chr(97 + i)}{j + 1}" for i in range(len(pick_B)) for j in range(3)])
fig.suptitle("From the fixed cost of fake_spectra to the pass of SkewerJAX: where the time goes", fontsize=21)
plt.show()
r128 = MEM.get(key(128, S.CPU_DEF))
rec = R["TNG50-2_n128_dv1-2-4-8-16"]
Nsk, nb_ = rec["config"]["n_skewers"], rec["grids"]["1"]["nbins"]
n_cr = sum(t["n_pieces"] for t in tiles128) / Nsk
mid_pieces = C2["midpoint"]["0.125"]["pieces_per_skewer"]
ev_w = {ln: ev["windows (+ one FFT)"][ln] for ln in S.LINE_IDS}
rows7 = [
    ("bytes read [GB per skewer]", (MEM[key(128, S.FS)]["read_GB"] if key(128, S.FS) in MEM else np.nan) / Nsk,
     (r128["read_GB"] if r128 else np.nan) / Nsk, "one pass, the rest of the rows only where needed"),
    ("cells whose physics is computed", 2 * rec["snapshot"]["n_gas"] / Nsk, rec["pass"]["n_needed"] / Nsk, "the cells within reach, once"),
    ("pieces (contributions)", mid_pieces, n_cr, "crossings instead of pieces sampled every 0.125 km/s"),
    ("profile evaluations (Lya + Si III)", 2 * mid_pieces * nb_, sum(ev_w.values()), "windows + FFT instead of the dense sum, on the crossings"),
]
print(f"\nper TNG50-2 skewer at 128²: {'naive':>12s} {'SkewerJAX':>12s} {'saved':>8s}")
for name, a, b, why in rows7:
    print(f"  {name:36s} {a:12.4g} {b:12.4g} {a / b:7.1f}x  ({why})")

*Reading.* At 16 skewers fake_spectra's bars are its fixed cost (I/O and gas physics); SkewerJAX's are its pass, mostly
the read of the geometry. At 1,024 skewers SkewerJAX reads every page once, still a fraction of fake_spectra's reads. At
$128^2$ and $256^2$ the tiles grow, and the windowed engine keeps them below fake_spectra's C engine even in the matched
mode. The table chains the savings per skewer: one pass instead of a read per line, the physics of the needed cells
instead of every cell, the crossings instead of sampled pieces, and windows plus an FFT instead of every pixel.

**The chain of savings.** Read once (D1), the physics of the cells within reach (D1), found by index arithmetic (D1);
one piece per crossing, its profile integrated in closed form (D2); each piece on its window, the far wings of all of
them in one FFT (D3); fake_spectra's own calculation on the same route (D4); fixed shapes, compiled once (D5). **The
limits** ([`docs/performance.md`](../docs/performance.md), §7): the disk for the pass, the host for the GPU entry, the
memory of the store for $512^2$ skewers (notebook 16).